# Step 6. Data Quality Documentation

**Project:** Seasonal Flight Delays at New York City Airports (2013)  
**Purpose:** Reproduce and document data-quality checks supporting the cleaned flight dataset and subsequent statistical analysis.

This notebook **does not overwrite** raw or cleaned data. It saves quality-audit tables to `results/tables/` and a documentation summary to `results/`.

**Scope:** The checks below document the actual cleaning logic from Step 3. They do not imply that missing arrival delays are exclusively cancellations or that extreme delays are errors.

## 1. Load raw and cleaned datasets

One row represents a flight in `flights.csv` and in the analysis-ready dataset. Weather records are hourly observations indexed by departure airport and `time_hour`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

raw_dir = Path("data/raw")
clean_path = Path("data/cleaned/analysis_ready_flights.csv")
table_dir = Path("results/tables")
report_dir = Path("results")
table_dir.mkdir(parents=True, exist_ok=True)

flights = pd.read_csv(raw_dir / "flights.csv")
weather = pd.read_csv(raw_dir / "weather.csv")
airlines = pd.read_csv(raw_dir / "airlines.csv")
cleaned = pd.read_csv(clean_path)

inventory = pd.DataFrame([
    {"Dataset": "Raw flights", "Rows": len(flights), "Columns": flights.shape[1]},
    {"Dataset": "Raw weather", "Rows": len(weather), "Columns": weather.shape[1]},
    {"Dataset": "Raw airlines", "Rows": len(airlines), "Columns": airlines.shape[1]},
    {"Dataset": "Analysis-ready flights", "Rows": len(cleaned), "Columns": cleaned.shape[1]},
])
display(inventory)
assert len(flights) == 336776
assert len(cleaned) == 327346
assert cleaned.shape[1] == 34

,Dataset,Rows,Columns
0,Raw flights,336776,19
1,Raw weather,26115,15
2,Raw airlines,16,2
3,Analysis-ready flights,327346,34


## 2. Missing values and inclusion decisions

The primary outcome is `arr_delay`. Flights without an observed arrival delay cannot be used in an arrival-delay analysis. Weather missingness is documented rather than filled or deleted in Step 3.

In [2]:
missing_outcome = int(flights["arr_delay"].isna().sum())
expected_clean = flights["arr_delay"].notna().sum()
assert expected_clean == len(cleaned)
assert cleaned["arr_delay"].notna().all()

weather_vars = ["temp", "dewp", "humid", "wind_speed", "precip", "visib"]
missing_audit = pd.DataFrame({
    "Variable": weather_vars,
    "Missing_Count": [int(cleaned[c].isna().sum()) for c in weather_vars],
    "Missing_Pct": [100 * cleaned[c].isna().mean() for c in weather_vars],
    "Valid_Count": [int(cleaned[c].notna().sum()) for c in weather_vars]
})

inclusion_audit = pd.DataFrame([
    {"Stage": "Original flight records", "Flights": len(flights)},
    {"Stage": "Excluded: missing arrival delay", "Flights": missing_outcome},
    {"Stage": "Retained: observed arrival delay", "Flights": len(cleaned)},
])
print(f"Missing arrival delay: {missing_outcome:,} ({100*missing_outcome/len(flights):.2f}%)")
display(inclusion_audit)
display(missing_audit.round(2))

Missing arrival delay: 9,430 (2.80%)


,Stage,Flights
0,Original flight records,336776
1,Excluded: missing arrival delay,9430
2,Retained: observed arrival delay,327346


,Variable,Missing_Count,Missing_Pct,Valid_Count
0,temp,1544,0.47,325802
1,dewp,1544,0.47,325802
2,humid,1544,0.47,325802
3,wind_speed,1605,0.49,325741
4,precip,1527,0.47,325819
5,visib,1527,0.47,325819


## 3. Duplicate records and join-key integrity

Check exact duplicate flight rows and whether the weather join keys (`origin`, `time_hour`) are unique. This matters because a many-to-many merge could artificially multiply flight records.

In [3]:
duplicate_audit = pd.DataFrame([
    {"Check": "Exact duplicate raw flight rows", "Count": int(flights.duplicated().sum())},
    {"Check": "Duplicate weather origin/time_hour keys", "Count": int(weather.duplicated(["origin", "time_hour"]).sum())},
    {"Check": "Duplicate airline carrier keys", "Count": int(airlines.duplicated(["carrier"]).sum())},
])
display(duplicate_audit)
assert not weather.duplicated(["origin", "time_hour"]).any(), "Weather keys are not unique."
assert not airlines.duplicated(["carrier"]).any(), "Airline carrier keys are not unique."

,Check,Count
0,Exact duplicate raw flight rows,0
1,Duplicate weather origin/time_hour keys,0
2,Duplicate airline carrier keys,0


## 4. Reproduce and validate the weather join

Step 3 used a **left join** on `origin` and `time_hour` with `validate="many_to_one"`. This check reproduces the join without modifying the original files. The `time_hour` key is the flight dataset's hourly timestamp; it should not be described as `scheduled_hour`.

In [4]:
flight_join = flights.copy()
weather_join = weather.copy()
flight_join["time_hour"] = pd.to_datetime(flight_join["time_hour"], utc=True)
weather_join["time_hour"] = pd.to_datetime(weather_join["time_hour"], utc=True)

joined = flight_join.merge(
    weather_join, on=["origin", "time_hour"], how="left",
    validate="many_to_one", indicator=True, suffixes=("", "_weather")
)
assert len(joined) == len(flights)

join_audit = (joined["_merge"].value_counts()
              .rename_axis("Merge_Status").reset_index(name="Flights"))
join_audit["Percentage"] = 100 * join_audit["Flights"] / len(joined)
display(join_audit.round(2))
print("Unmatched records remain in the left-joined dataset.")

,Merge_Status,Flights,Percentage
0,both,335220,99.54
1,left_only,1556,0.46
2,right_only,0,0.00


Unmatched records remain in the left-joined dataset.


## 5. Derived variables and cleaned-dataset consistency

Recompute `season` from month and `scheduled_hour` from `sched_dep_time` to confirm that the saved analysis file matches the documented transformations. Use row order only after confirming that the retained raw flight records match the saved flight identifiers.

In [5]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
}
checks = {
    "No missing arrival delay": cleaned["arr_delay"].notna().all(),
    "All seasons valid": cleaned["season"].isin(season_map.values()).all(),
    "Season matches month": cleaned["season"].eq(cleaned["month"].map(season_map)).all(),
    "Scheduled hour in 0-23": cleaned["scheduled_hour"].between(0, 23).all(),
    "Scheduled hour matches schedule": cleaned["scheduled_hour"].eq(
        cleaned["sched_dep_time"] // 100).all(),
    "No missing distance": cleaned["distance"].notna().all(),
    "Expected origin airports": cleaned["origin"].isin(["EWR", "JFK", "LGA"]).all(),
    "No saved merge indicator": "_merge" not in cleaned.columns,
}
validation_audit = pd.DataFrame({"Check": list(checks), "Passed": list(checks.values())})
display(validation_audit)
assert validation_audit["Passed"].all(), "Review failed data validation checks."

,Check,Passed
0,No missing arrival delay,True
1,All seasons valid,True
2,Season matches month,True
3,Scheduled hour in 0-23,True
4,Scheduled hour matches schedule,True
5,No missing distance,True
6,Expected origin airports,True
7,No saved merge indicator,True


## 6. Weather plausibility checks

These checks identify impossible negative measurements and out-of-range humidity. They do not establish that every remaining weather value is accurate.

In [6]:
plausibility = pd.DataFrame([
    {"Check": "Humidity outside 0-100%", "Flagged": int((cleaned["humid"].notna() & ~cleaned["humid"].between(0, 100)).sum())},
    {"Check": "Negative wind speed", "Flagged": int((cleaned["wind_speed"] < 0).sum())},
    {"Check": "Negative precipitation", "Flagged": int((cleaned["precip"] < 0).sum())},
    {"Check": "Negative visibility", "Flagged": int((cleaned["visib"] < 0).sum())},
])
display(plausibility)
if plausibility["Flagged"].sum():
    print("Review flagged values; do not silently remove them.")

,Check,Flagged
0,Humidity outside 0-100%,0
1,Negative wind speed,0
2,Negative precipitation,0
3,Negative visibility,0


## 7. Arrival-delay outliers and sensitivity

Flag IQR outliers without dropping observations. Quantify how the mean changes when unusually large positive delays are omitted **only for a sensitivity check**.

In [7]:
d = cleaned["arr_delay"]
q1, q3 = d.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
outlier_n = int(((d < low) | (d > high)).sum())
outlier_audit = pd.DataFrame([{
    "Q1": q1, "Q3": q3, "IQR": iqr,
    "Lower_Bound": low, "Upper_Bound": high,
    "Flagged_Outliers": outlier_n,
    "Outlier_Pct": 100*outlier_n/len(d),
    "Mean": d.mean(), "Median": d.median(), "Skewness": d.skew()
}])
sensitivity_audit = pd.DataFrame([
    {"Scenario": "All flights", "Flights": len(d), "Mean": d.mean(), "Median": d.median()},
    *[{"Scenario": f"Exclude positive delays > {cutoff} min",
       "Flights": int((d <= cutoff).sum()), "Mean": d[d <= cutoff].mean(),
       "Median": d[d <= cutoff].median()} for cutoff in (180, 120, 60)]
])
display(outlier_audit.round(3))
display(sensitivity_audit.round(2))
assert outlier_n == 27880

,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Flagged_Outliers,Outlier_Pct,Mean,Median,Skewness
0,-17.0,14.0,31.0,-63.5,60.5,27880,8.517,6.895,-5.0,3.717


,Scenario,Flights,Mean,Median
0,All flights,327346,6.90,-5.0
1,Exclude positive delays > 180 min,323503,4.02,-5.0
2,Exclude positive delays > 120 min,317312,1.26,-6.0
3,Exclude positive delays > 60 min,299557,-3.71,-7.0


## 8. Regression-readiness and remaining limitations

Compare complete-case sample sizes for a flight-only specification and a specification with selected weather predictors. The comparison is a **data-quality check**, not model fitting. For a fair model comparison in Milestone 4, fit both models on the **same complete-case rows**.

In [8]:
flight_predictors = ["arr_delay", "season", "carrier", "origin", "scheduled_hour", "distance"]
weather_predictors = flight_predictors + ["temp", "humid", "wind_speed", "precip", "visib"]
model_readiness = pd.DataFrame([
    {"Specification": "Flight-only", "Complete_Cases": int(cleaned[flight_predictors].dropna().shape[0])},
    {"Specification": "Flight + weather", "Complete_Cases": int(cleaned[weather_predictors].dropna().shape[0])},
])
model_readiness["Share_of_Analysis_Sample_Pct"] = 100 * model_readiness["Complete_Cases"] / len(cleaned)
display(model_readiness.round(2))
assert model_readiness.loc[0, "Complete_Cases"] == 327346
assert model_readiness.loc[1, "Complete_Cases"] == 325724

,Specification,Complete_Cases,Share_of_Analysis_Sample_Pct
0,Flight-only,327346,100.0
1,Flight + weather,325724,99.5


## 9. Export quality-audit tables

In [9]:
exports = {
    "quality_dataset_inventory": inventory,
    "quality_inclusion_audit": inclusion_audit,
    "quality_weather_missingness": missing_audit,
    "quality_duplicate_audit": duplicate_audit,
    "quality_weather_join_audit": join_audit,
    "quality_validation_checks": validation_audit,
    "quality_weather_plausibility": plausibility,
    "quality_outlier_audit": outlier_audit,
    "quality_outlier_sensitivity": sensitivity_audit,
    "quality_model_readiness": model_readiness,
}
for name, frame in exports.items():
    frame.to_csv(table_dir / f"{name}.csv", index=False)
print(f"Saved {len(exports)} audit tables to {table_dir}")

Saved 10 audit tables to results/tables


## 10. Data Quality Documentation — Summary

### Data sources and unit of analysis
The analysis uses the 2013 `nycflights13` flights, weather, and airlines tables. Each retained row represents a flight departing from EWR, JFK, or LGA with an observed arrival delay.

### Cleaning and inclusion decisions
The original flights table contains **336,776** records. **9,430** flights have missing `arr_delay` and were excluded from arrival-delay analysis, leaving **327,346** flights and **34** saved columns. The absence of `arr_delay` alone does not identify the precise reason for each missing observation. The raw files are retained unchanged.

### Join integrity and transformations
Hourly weather observations were left-joined on `origin` and `time_hour` with a many-to-one validation. The join preserves all original flight rows; unmatched flights remain present with missing weather fields. `season` was derived from month, and `scheduled_hour` from the scheduled departure clock time (`sched_dep_time // 100`).

### Missingness and outliers
The final dataset has no missing `arr_delay`. Weather measurements retain approximately **0.47%–0.49%** missing values per variable. The IQR rule flagged **27,880 (8.52%)** arrival-delay observations as statistically unusual; these were **not removed**. The right-skewed arrival-delay distribution (skewness approximately **3.72**) and extreme positive delays warrant attention during modeling.

### Readiness for statistical analysis
All **327,346** retained flights have complete flight-only predictors in the proposed specification. **325,724** have complete selected flight and weather predictors. Multivariable model comparisons should use the same rows to separate predictor improvements from sample-composition changes.

### Remaining limitations
- Observations cover **2013 NYC-area departures only**, limiting generalization to current operations.
- Missing outcome records are excluded, so results describe flights with observed arrival delays, not necessarily all scheduled flights.
- Hourly weather at the departure airport may not represent conditions at the destination or along the route.
- Multiple flights may share one hourly weather observation, which matters for statistical uncertainty.
- Extreme delays may be genuine observations rather than recording errors; avoid deleting them without a documented reason.
- Data-quality checks establish internal consistency, not causal interpretation or freedom from measurement error.